# Build Features from collection.db

**Run this first.** Loads candles + snapshots from SQLite, computes 60 technical indicators
(including volume), saves to `data/latest_features.jsonl`. All other notebooks consume this file.

**Pipeline:**
1. Load candles + snapshots from collection.db
2. Detect contiguous sessions (gaps > 5 min)
3. Skip first 21 candles per session (indicator warm-up)
4. Compute 60 indicators per snapshot using typed models
5. Save to `data/latest_features.jsonl`

In [ ]:
import json
import sqlite3
from pathlib import Path

import pandas as pd
from technicals import CandleRecord, IndicatorSnapshot, compute_all
from tqdm import tqdm

DB_PATH = Path("../data/collection.db")
FEATURES_PATH = Path("../data/latest_features.jsonl")
WARM_UP = 21

## 1. Load from SQLite

In [ ]:
conn = sqlite3.connect(str(DB_PATH))
candles_df = pd.read_sql("SELECT * FROM candles ORDER BY start_time", conn)
snaps_df = pd.read_sql("SELECT * FROM snapshots ORDER BY candle_id, timestamp", conn)
conn.close()

# Detect sessions (gaps > 5 min)
candles_df = candles_df.sort_values("start_time").reset_index(drop=True)
candles_df["gap"] = candles_df["start_time"] - candles_df["end_time"].shift(1)
candles_df["session"] = (candles_df["gap"] > 300).cumsum()
candles_df["session_idx"] = candles_df.groupby("session").cumcount()
candles_df["usable"] = candles_df["session_idx"] >= WARM_UP

for sid, grp in candles_df.groupby("session"):
    total = len(grp)
    usable = grp["usable"].sum()
    print(f"Session {sid}: {total} candles, {usable} usable")

print(f"\nTotal: {len(candles_df)} candles, {candles_df['usable'].sum()} usable")
print(f"Snapshots: {len(snaps_df):,}")

## 2. Compute indicators

In [ ]:
all_rows = []

for sid, session_candles in candles_df.groupby("session"):
    prior_candles: list[CandleRecord] = []

    for _, cr in tqdm(session_candles.iterrows(), total=len(session_candles), desc=f"Session {sid}"):
        cid = cr["candle_id"]
        is_usable = cr["usable"]

        candle = CandleRecord(
            candle_id=cid,
            start_time=cr["start_time"],
            end_time=cr["end_time"],
            open=cr["open"],
            high=cr["high"],
            low=cr["low"],
            close=cr["close"],
            volume=cr["volume"],
            outcome=cr["outcome"],
            final_ret=cr["final_ret"],
        )

        if is_usable:
            snap_rows = snaps_df[snaps_df["candle_id"] == cid]
            if len(snap_rows) < 5:
                prior_candles.append(candle)
                continue

            snapshots = []
            for _, s in snap_rows.iterrows():
                ob = json.loads(s["orderbook_json"])
                snapshots.append(
                    IndicatorSnapshot(
                        candle_id=cid,
                        timestamp=s["timestamp"],
                        elapsed_pct=s["elapsed_pct"],
                        btc_price=s["btc_price"],
                        btc_bid=s["btc_bid"],
                        btc_ask=s["btc_ask"],
                        up_bids=[ob["up_bids"][0]] if ob.get("up_bids") else [],
                        up_asks=[ob["up_asks"][0]] if ob.get("up_asks") else [],
                        down_bids=[ob["down_bids"][0]] if ob.get("down_bids") else [],
                        down_asks=[ob["down_asks"][0]] if ob.get("down_asks") else [],
                        market_volume=s["market_volume"],
                    )
                )

            for si in range(len(snapshots)):
                indicators = compute_all(prior_candles, candle.open, snapshots[: si + 1])
                snap = snapshots[si]
                row = {
                    "candle_id": cid,
                    "session": int(sid),
                    "timestamp": snap.timestamp,
                    "elapsed_pct": snap.elapsed_pct,
                    "btc_price": snap.btc_price,
                    "up_best_bid": snap.up_bids[0][0] if snap.up_bids else None,
                    "up_best_ask": snap.up_asks[0][0] if snap.up_asks else None,
                    "up_bid_depth": snap.up_bids[0][1] if snap.up_bids else None,
                    "up_ask_depth": snap.up_asks[0][1] if snap.up_asks else None,
                    "down_best_bid": snap.down_bids[0][0] if snap.down_bids else None,
                    "down_best_ask": snap.down_asks[0][0] if snap.down_asks else None,
                    "down_bid_depth": snap.down_bids[0][1] if snap.down_bids else None,
                    "down_ask_depth": snap.down_asks[0][1] if snap.down_asks else None,
                    "market_volume": snap.market_volume,
                    **indicators,
                    "outcome": candle.outcome,
                }
                all_rows.append(row)

        prior_candles.append(candle)

df = pd.DataFrame(all_rows)
print(f"\nTotal rows: {len(df):,} from {df['candle_id'].nunique()} candles")
print(f"UP rate: {(df['outcome'] == 'UP').mean() * 100:.1f}%")

## 3. Save to JSONL

In [ ]:
with open(FEATURES_PATH, "w") as f:
    for _, row in df.iterrows():
        f.write(json.dumps({k: v if pd.notna(v) else None for k, v in row.items()}) + "\n")

print(f"Saved {len(df):,} rows to {FEATURES_PATH}")
print(f"File size: {FEATURES_PATH.stat().st_size / 1024 / 1024:.1f} MB")